# CDI Layer 2 and Layer 3

Use the **compute** kernel. After repository code changes, restart the kernel once. Put `OPENAI_API_KEY` in the repository `.env`, set the fact-sheet path and Layer 2 reasoning effort below, and run this cell. It creates a resumable eight-mission Layer 2 run and shows the recorded usage plus one mission for inspection.

In [ ]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from IPython.display import JSON, Markdown, display

from ML.deep_research.layer2.cli import load_dotenv_key, run_all as run_layer2
from ML.deep_research.layer2.create_run import create_run as create_layer2_run
from ML.deep_research.layer2.fs import load_json
from ML.deep_research.layer2.settings import PLANNER_PATH, RUNS_DIR

FACT_SHEET_PATH = Path(r"inputs\1_Amtsgericht_Bad-Homburg_e0b0ca5b_fact_sheet.md")
LAYER2_REASONING_EFFORT = "medium"  # low | medium | high | max
PREVIEW_MISSION = 0

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_RUN = create_layer2_run(
    FACT_SHEET_PATH,
    PLANNER_PATH,
    RUNS_DIR,
    reasoning_effort=LAYER2_REASONING_EFFORT,
)
print(f"Layer 2 run created: {L2_RUN}")
print(f"Resume if interrupted: .\\run.ps1 -Resume '{L2_RUN}'")
L2_USAGE = await asyncio.to_thread(run_layer2, L2_RUN)

mission_files = sorted((L2_RUN / "missions").glob("*.json"))
display(Markdown("### Recorded Layer 2 usage"))
display(JSON(data=L2_USAGE, expanded=True))
print(f"Mission files: {len(mission_files)}")
for path in mission_files:
    print(f"- {path.name}")
if mission_files:
    preview = mission_files[PREVIEW_MISSION]
    display(Markdown(f"### Mission preview: `{preview.name}`"))
    display(JSON(data=load_json(preview), expanded=False))


## Layer 3 — live online research

Run this only after Layer 2 completes. **This cell confirms that the input is public or invented, sends research queries to external services, and consumes model/web-search usage.** Set the exact Layer 2 run path, Layer 3 model reasoning, web-search depth, and web-search verbosity below. A blank run path uses `L2_RUN` from the Layer 2 cell.

In [2]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.cli import load_dotenv_key
from ML.deep_research.layer2.fs import load_json, read_text
from ML.deep_research.layer3.cli import run_all as run_layer3
from ML.deep_research.layer3.pipeline.create_run import create_run as create_layer3_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR, SCHEMA_VERSION
from ML.deep_research.layer3.usage import summarize_usage

LAYER3_SOURCE_RUN_PATH = r""  # paste a runs/.../L2_* folder; blank uses L2_RUN above
LAYER3_MODEL_REASONING_EFFORT = "low"  # low | medium | high | max
WEB_SEARCH_DEPTH = "low"  # low | medium | high
WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
PUBLIC_INPUT_CONFIRMED = True

load_dotenv_key()
if LAYER3_SOURCE_RUN_PATH.strip():
    L2_RUN = Path(LAYER3_SOURCE_RUN_PATH)
elif globals().get("L2_RUN"):
    L2_RUN = Path(L2_RUN)
else:
    raise RuntimeError("Set LAYER3_SOURCE_RUN_PATH to the exact Layer 2 run folder.")
print(f"Using existing Layer 2 run: {L2_RUN}")

if not PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 3 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")

for candidate in sorted(
    LAYER3_RUNS_DIR.rglob("L3_*"), key=lambda path: path.stat().st_mtime, reverse=True
):
    candidate_record = load_json(candidate / "run.json")
    source_path = candidate_record.get("source_l2", {}).get("path", "")
    search_options = candidate_record.get("web_search", {})
    if (
        candidate_record.get("schema_version") == SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L2_RUN.resolve()
        and candidate_record.get("reasoning_effort") == LAYER3_MODEL_REASONING_EFFORT
        and search_options.get("context_size") == WEB_SEARCH_DEPTH
        and search_options.get("verbosity") == WEB_SEARCH_VERBOSITY
    ):
        L3_RUN = candidate
        break
else:
    L3_RUN = create_layer3_run(
        L2_RUN,
        LAYER3_RUNS_DIR,
        public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER3_MODEL_REASONING_EFFORT,
        web_search_context_size=WEB_SEARCH_DEPTH,
        web_search_verbosity=WEB_SEARCH_VERBOSITY,
    )

l3_before = load_json(L3_RUN / "run.json")
execution = l3_before.get("execution", {})
resume_command = f".\\run.ps1 -ResumeL3 '{L3_RUN}'"
print(f"Layer 3 run: {L3_RUN}")
print(f"Resume if interrupted: {resume_command}")

layer3_task = None
if l3_before.get("status") != "complete":
    layer3_task = asyncio.create_task(run_layer3(L3_RUN))
while layer3_task and not layer3_task.done():
    await asyncio.sleep(5)
    live = load_json(L3_RUN / "run.json")
    domains = live.get("execution", {}).get("domains", {})
    running = [name for name, item in domains.items() if item.get("status") == "running"]
    completed = [name for name, item in domains.items() if item.get("status") == "complete"]
    domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
    lines = read_text(L3_RUN / "usage.jsonl").splitlines()
    try:
        latest = json.loads(lines[-1]) if lines else {}
    except json.JSONDecodeError:
        latest = {}
    clear_output(wait=True)
    print(f"Layer 3 run: {L3_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Active coordinator: {running[0] if running else 'none'}")
    print(f"Completed coordinators: {len(completed)}/8")
    print(f"Saved domain responses: {len(domain_finals)}/8")
    print("Usage:", summarize_usage(L3_RUN))
    if latest:
        print("Last activity:", latest.get("timestamp"), latest.get("actor"), latest.get("phase"), latest.get("detail", ""))

L3_ERROR = ""
if layer3_task:
    try:
        await layer3_task
    except Exception as error:
        L3_ERROR = f"{type(error).__name__}: {error}"
clear_output(wait=True)

l3_record = load_json(L3_RUN / "run.json")
domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
final_answer = L3_RUN / "research" / "final.md"
final_records = [*l3_record.get("execution", {}).get("domains", {}).values(), l3_record.get("execution", {}).get("final", {})]
print(f"Layer 3 run: {L3_RUN}")
print(f"Status: {l3_record.get('status', 'unknown')}")
print(f"Resume if interrupted: .\\run.ps1 -ResumeL3 '{L3_RUN}'")
if L3_ERROR:
    print(f"Execution error: {L3_ERROR}")
failed = [
    f"{item.get('stage')}: {item.get('actor')} — {item.get('error')}"
    for item in final_records
    if item.get("status") == "failed"
]
for item in failed:
    print(f"Failed stage: {item}")
display(Markdown("### Recorded Layer 3 usage"))
display(JSON(data=summarize_usage(L3_RUN), expanded=True))
print(f"Saved domain responses: {len(domain_finals)}/8")
for path in domain_finals:
    print(f"- {path.relative_to(L3_RUN)}")
display(Markdown("### Property synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the incomplete Layer 3 run shown above."))


Layer 3 run: C:\Users\Arjun Gowda\Desktop\Sed_ai_v2\runs\inputs-1-amtsgericht-bad-homburg-e0b0ca5b-fact-sheet-0d62dba4\L3_20260821_182919_b675
Status: complete
Resume if interrupted: .\run.ps1 -ResumeL3 'C:\Users\Arjun Gowda\Desktop\Sed_ai_v2\runs\inputs-1-amtsgericht-bad-homburg-e0b0ca5b-fact-sheet-0d62dba4\L3_20260821_182919_b675'


### Recorded Layer 3 usage

<IPython.core.display.JSON object>

Saved domain responses: 8/8
- domains\asset-integrity-systems-and-operational-resilience\final.md
- domains\energy-carbon-and-transition\final.md
- domains\external-dependencies-geopolitics-trade-and-supply-chains\final.md
- domains\finance-debt-and-macro-transmission\final.md
- domains\ground-physical-climate-and-insurability\final.md
- domains\location-demand-market-valuation-and-exit\final.md
- domains\occupier-lease-income-and-counterparty-economics\final.md
- domains\rights-public-law-and-ownership-governance\final.md


### Property synthesis

# Property risk landscape

## Executive risk picture

The supplied responses identify a courthouse at Auf der Steinkaut 10–12, Bad Homburg, currently operated as an Amtsgericht. The material property-linked risks are predominantly unresolved technical, legal, contractual and evidentiary exposures rather than proven closure, default, contamination, structural instability or current statutory prohibition.

The most material current exposures are:

- incomplete assurance of fire doors, escape routes, smoke extraction, fire alarm, emergency lighting, electrical protection, fire-water separation and inspection close-out;
- underground-garage deterioration, access-gate failure, uncertain ventilation/CO performance, drainage and backflow protection, and potential surface-water or groundwater entry;
- incomplete works, conflicting scopes and costs, uncertain contractor delivery, specialist-equipment obsolescence and unverified acceptance;
- unresolved title, cadastral, access, parking, lawful-use, fire-safety and ownership-authority records;
- contradictory heating records, an unverified energy baseline and conditional future exposure to heating, automation, charging and refrigerant requirements;
- unverified lease term, break and renewal rights, landlord identity, repair allocation, recoveries and counterparty chain;
- conditional access and operational disruption from nearby parking restrictions and dated transport works; and
- potential reletting, valuation, funding and exit uncertainty if judicial occupation changes.

External market, climate, water-supply, construction-price, lending and labour evidence supports the plausibility of several pathways but does not establish a property-specific loss, financing event, insurance denial, contractor failure or value effect. Current court operation is contrary evidence against an established total-use prohibition, but it does not resolve the underlying documentation gaps.

All researchers used the same model and runtime. Agreement between domain responses is therefore not independent confirmation or field-wide consensus.

## Current and upcoming nearby developments

- **Seedammbad/Seedammweg parking restriction — April–June 2026:** A fire was reported at the Seedammbad parking garage on 23 April 2026. Police reporting indicated continued closure during structural assessment, and a City notice stated that 405 spaces partially reopened from 19 June 2026 with restrictions to bath and spa users. The court directions page continues to describe free nearby Seedammweg parking, creating unresolved evidence about whether the facilities are the same and whether the alternative remains available to court users.  
  Sources: [Police report](https://www.presseportal.de/blaulicht/pm/50152/6261635); [City partial reopening notice](https://www.bad-homburg.de/de/stadt/aktuelles/seedammbad-parkhaus-oeffnet-in-teilen-wieder-jy8bprzynx); [Municipal parking overview](https://app.bad-homburg.de/de/erleben/informieren/parken/uebersicht-parkhaeuser); [Official court directions](https://ordentliche-gerichtsbarkeit.hessen.de/landgerichtsbezirk-frankfurt-am-main/amtsgericht-bad-homburg-v-d-dhoehe/wegbeschreibung)

- **Barrier-free bus-stop and access works around the Amtsgericht — remainder of 2026:** The City announced planned works around the Amtsgericht and Kaiserin-Friedrich-Gymnasium involving paving, curbs, roadway and footway reconstruction. The supported pathway is temporary change to pedestrian, bus and approach routes; court-specific phasing and emergency-route effects are not established.  
  Source: [City accessibility works announcement](https://www.bad-homburg.de/de/stadt/aktuelles/stadt-baut-bushaltestellen-kontinuierlich-aus-y65gqbyakk)

- **U2 extension and related transport changes — 2025–2029:** Construction began with a December 2025 groundbreaking. Project material describes works through 2028 and planned passenger operation in 2029; earlier municipal material referred to 2030. Bus-route changes were announced from 16 March 2026. The supported pathway is temporary access, wayfinding and journey-time friction for court users, staff, deliveries and emergency access; parcel-specific effects remain unknown.  
  Sources: [U2 project timeline](https://bad-homburg-u2.de/das-projekt.html); [City groundbreaking notice](https://www.bad-homburg.de/de/stadt/aktuelles/spatenstich-fuer-die-verlaengerung-der-u2-oz5mnkkyvr); [City bus changes](https://www.bad-homburg.de/de/stadt/aktuelles/ab-16-maerz-aenderungen-im-stadtbusverkehr-kq8vpdgy26); [City traffic notices](https://www.bad-homburg.de/de/stadt/aktuelles/verkehrsmeldungen-wbaq02ayw2)

- **Local court operating changes — from 1 January 2026:** Electronic files were reported for introduction across Hessian courts, and the Amtsgerichte civil-jurisdiction threshold was increased to €10,000 under current §23 GVG. These are institutional rather than physical developments, but they have a supported path to archive, hearing-room, staffing and space-use requirements at the courthouse. The direction and scale of the property effect are unknown.  
  Sources: [Hessen Justice e-file announcement](https://justizministerium.hessen.de/presse/e-akte-erfolgreich-in-der-justiz-eingefuehrt); [Hessen Justice jurisdiction announcement](https://vg-frankfurt.justiz.hessen.de/presse/zustaendigkeiten-der-amtsgerichte-werden-erweitert); [Current §23 GVG](https://www.gesetze-im-internet.de/gvg/__23.html)

## Cross-domain risk register

### R1. Fire doors, compartmentation and escape-route assurance

- **New evidence:** ArbStättV requires safety installations, escape routes and emergency exits to be maintained and tested; ASR A2.3 addresses escape-route arrangement, dimensions, doors and safety lighting.
- **Existing property fact:** Records report defective or incomplete fire-door items, including doors 28, 46, 13 and 15, obsolete or missing components, outdated or absent escape plans and incomplete testing and defect close-out.
- **Property linkage:** Defective or uncertified fire doors and escape routes may permit smoke spread, delay evacuation or restrict affected areas in a continuously occupied public courthouse.
- **Likelihood / impact:** High / High.
- **Affected:** Court users, staff, contractors, emergency responders, affected routes and areas, repair and re-testing costs.
- **Time horizon / status:** Current throughout the hold; unresolved defect and evidence exposure, not a verified prohibition on use.
- **Missing evidence:** Complete door register; fire-safety concept; door ratings and locations; current certificates; defect close-out; acceptance evidence.
- **Sources:** [ArbStättV §4](https://www.gesetze-im-internet.de/arbst_ttv_2004/__4.html); [BAuA ASR A2.3](https://www.baua.de/DE/Angebote/Regelwerk/ASR/ASR-A2-3.html); supplied TDD, inspection and maintenance records.

### R2. Smoke extraction, RWA and roof-window controls

- **New evidence:** HBO §38 addresses smoke removal from necessary stairwells; the Hessian TPrüfV covers applicable RWA and safety systems and recurring effectiveness and operational-safety examinations.
- **Existing property fact:** Stairwells A and C have unresolved RWA requirements, possible inadequate area, overdue or incomplete maintenance evidence and missing power-supply labels. One 2025 report describes functional and safe continued operation; another reports material safety concerns and reinspection.
- **Property linkage:** Inadequate opening area, activation, controls, power or roof-window operation could allow smoke to remain in escape or firefighting routes.
- **Likelihood / impact:** Medium / High.
- **Affected:** Stairwells, occupants, fire services and court operations; possible restrictions, reinspection and remedial cost.
- **Time horizon / status:** Current; materially unresolved contradiction and compliance evidence gap.
- **Missing evidence:** Approved fire-safety concept; required RWA area; complete signed report; cause-and-effect matrix; free-area, opening, power and battery tests; reinspection and close-out; TPrüfV applicability.
- **Sources:** [HBO reading version](https://wirtschaft.hessen.de/sites/wirtschaft.hessen.de/files/2025-12/hbo_broschuere_stand_oktober_2025-baupaket_i-final_nach_korrektur_vom_27.11.2025.pdf); [Hessian TPrüfV](https://www.rv.hessenrecht.hessen.de/bshe/?query=DOKNR%3Ajlr-NNLHE00005472&source=PermaLink); [H‑VV TB](https://wirtschaft.hessen.de/wohnen-bauen/bauvorschriften/technische-baubestimmungen); supplied RWA records.

### R3. Fire alarm, alarm audibility and transmission dependency

- **New evidence:** ArbStättV requires fire-alarm and safety systems to be maintained and tested. Local fire-alarm connection conditions identify the Hochtaunuskreis receiving centre and connection, acceptance and commissioning requirements. EN 50136-related transmission and telecommunications migration are relevant external dependencies.
- **Existing property fact:** Records describe partial fire-alarm protection, possible inadequate audibility in rooms 103 and 105, ageing detectors and control equipment, conflicting findings on whether the alarm is functional, and inconsistent descriptions of the telecommunications route and receiving endpoint.
- **Property linkage:** Delayed warning or a failed route to the public receiving centre could impair evacuation and fire-service response; obsolete equipment may increase replacement time and cost.
- **Likelihood / impact:** Medium / High for system performance; Unknown for transmission failure.
- **Affected:** Occupants, emergency responders, alarm and communications systems, court continuity and replacement expenditure.
- **Time horizon / status:** Current and near-term; emerging system-performance and transmission-assurance exposure, not a verified failure.
- **Missing evidence:** Approved coverage and audibility concept; measurements; equipment model and age; current receiving-centre identity; route technology and redundancy; PTI34 feasibility; end-to-end tests; acceptance records.
- **Sources:** [ArbStättV §4](https://www.gesetze-im-internet.de/arbst_ttv_2004/__4.html); [Hochtaunuskreis control centre](https://www.hochtaunuskreis.de/Landkreis/Gefahrenabwehr/Zentrale-Leitstelle/); [Bad Homburg fire-alarm connection conditions](https://www.uds-beratung.de/tl_files/pdf/UDS%20Download/TAB-Feuerwehren/TAB%2061350%20Bad%20Homburg%202021-12-01.pdf); [DIN transmission information](https://www.din.de/de/mitwirken/normenausschuesse/fnfw/uebertragung-und-fernzugriff-auf-brandmeldeanlagen-1225634); [Bundesnetzagentur migration](https://www.bundesnetzagentur.de/SharedDocs/Pressemitteilungen/DE/2024/20241119_Gigabit.html); supplied fire-alarm records.

### R4. Electrical protection, lightning protection and common safety-power dependency

- **New evidence:** ArbStättV and DGUV Vorschrift 4 support maintenance and recurring testing of safety and public-sector electrical installations.
- **Existing property fact:** Electrical inspection records are incomplete; lightning-protection and earthing defects, incomplete emergency-lighting records and missing RWA power labels are reported.
- **Property linkage:** A common electrical or emergency-power fault could affect fire alarm, RWA, emergency lighting, access, controls and communications simultaneously. Lightning or earthing defects could affect equipment and fire exposure.
- **Likelihood / impact:** Unknown / High.
- **Affected:** Occupants, emergency responders, technical systems and court continuity; testing, repair and outage costs.
- **Time horizon / status:** Current throughout the hold; high-consequence evidence gap, not a demonstrated common-mode failure.
- **Missing evidence:** Single-line diagrams; protective-device and earthing measurements; lightning certificates; emergency-power load and autonomy tests; current DGUV/VDE/VdS records; dependency map.
- **Sources:** [ArbStättV §4](https://www.gesetze-im-internet.de/arbst_ttv_2004/__4.html); [DGUV Vorschrift 4](https://publikationen.dguv.de/regelwerk/dguv-vorschriften/1457/elektrische-anlagen-und-betriebsmittel); supplied electrical and lightning-protection records.

### R5. Underground-garage ventilation and carbon monoxide

- **New evidence:** GaV §16 provides mechanical-exhaust, supply-air, CO, two-fan and automatic-changeover requirements. The supplied 7,860 m³/h figure equals approximately 6.0 m³/(h·m²) against cited deemed-compliance values of 8 or 16, if the area and figure are applicable and represent demonstrated capacity.
- **Existing property fact:** The garage is described as 1,310 m² with two fans, CO warning and staged control, but traffic classification, actual performance and inspection records are incomplete.
- **Property linkage:** Insufficient airflow, failed changeover, inadequate CO warning or loss of safety power could permit CO accumulation or impair smoke clearance.
- **Likelihood / impact:** Unknown / High.
- **Affected:** Garage users, staff, contractors, vehicles and emergency responders; possible garage closure and testing or replacement cost.
- **Time horizon / status:** Current throughout the hold; material capacity-verification gap, not established non-compliance.
- **Missing evidence:** Statutory usable area; traffic classification; permit and transition status; fan tests; CO readings and sensor map; alarm, changeover and safety-power tests; operating data.
- **Sources:** [GaV 2024](https://verkuendung.hessen.de/sites/verkuendung.hessen.de/files/veroeffentlichungsplattform_plugin/published/274/GVBl_2024_Nr_18_Regelungstext.pdf); [GaV amendment 2025](https://verkuendung.hessen.de/sites/verkuendung.hessen.de/files/veroeffentlichungsplattform_plugin/published/1217/GVBl_2025_Nr_90_Regelungstext.pdf); supplied garage ventilation records.

### R6. Garage concrete deterioration and structural-capacity uncertainty

- **New evidence:** DIBt technical rules require existing-condition assessment and justified maintenance planning; reinforcement corrosion is a recognized deterioration mechanism, but the cited sources do not provide property-specific calculations.
- **Existing property fact:** The TDD reports substantial concrete damage and reinforcement corrosion, a medium-term €17,000 wall/coating item and broader garage works bases of €765,000 and €702,668.06 net.
- **Property linkage:** Deterioration may progress to spalling, water ingress, local safety restrictions or reduced structural capacity. Present structural instability is not established.
- **Likelihood / impact:** Medium / High.
- **Affected:** Garage structure, users, vehicles, pedestrian and emergency access, repair CapEx and downtime.
- **Time horizon / status:** Current through the 10-year hold; established supplied defect with unverified progression and capacity.
- **Missing evidence:** Crack, spall and corrosion mapping; reinforcement-loss assessment; water-ingress history; structural calculations/load rating; repair design; completion and acceptance records.
- **Sources:** [DIBt TR Instandhaltung](https://www.dibt.de/de/aktuelles/meldungen/nachricht-detail/meldung/technische-regel-instandhaltung-von-betonbauwerken-neu-ab-januar-2021); [TÜV SÜD garage repair guidance](https://www.tuvsud.com/de-de/indust-re/bautechnik-info/sanierung/garagensanierung); supplied garage TDD and works records.

### R7. Garage gate, parking access and external parking continuity

- **New evidence:** The nearby parking facility was damaged by fire and partially reopened under restrictions. The court directions page continues to state that free on-site and nearby parking are available.
- **Existing property fact:** The subject garage roller gate is reported failed or immobile; parking counts, access rights and effective capacity conflict.
- **Property linkage:** Gate failure may independently restrict the subject garage. External parking restrictions may inconvenience court users and reduce parking redundancy, but direct on-site capacity and emergency-access impairment are not established.
- **Likelihood / impact:** Medium for subject access; High for the reported external restriction during the cited period / Medium for operational effect.
- **Affected:** Visitors, staff, judges, lawyers, witnesses, police, emergency vehicles and contractors; access delay and parking displacement.
- **Time horizon / status:** Subject gate current; external event reported April–June 2026 with continuation unknown; established events with inferred property effect.
- **Missing evidence:** Gate repair programme; current parking capacity and restrictions; confirmation that Seedammweg and Seedammbad identify the same facility; court on-site capacity; emergency-access records; parking demand and lease provisions.
- **Sources:** [Official court directions](https://ordentliche-gerichtsbarkeit.hessen.de/landgerichtsbezirk-frankfurt-am-main/amtsgericht-bad-homburg-v-d-hoehe/wegbeschreibung); [Police parking-garage report](https://www.presseportal.de/blaulicht/pm/50152/6261635); [City reopening notice](https://www.bad-homburg.de/de/stadt/aktuelles/seedammbad-parkhaus-oeffnet-in-teilen-wieder-jy8bprzynx); supplied garage-access records.

### R8. Lifts, accessibility and concentrated vertical-transport dependency

- **New evidence:** BetrSichV provides for recurring lift examinations, emergency communication and rescue arrangements. The official court page confirms a ramp and one passenger lift; supplied records identify two lift installations in another reading.
- **Existing property fact:** Lift risk-assessment and inspection evidence is unavailable; the property file also identifies one combined passenger, freight and apparent fire-service lift.
- **Property linkage:** A lift outage or undocumented condition could impair accessible public and staff access, deliveries, emergency response and court logistics, with no verified parallel lift.
- **Likelihood / impact:** Unknown / Medium to High.
- **Affected:** Persons with reduced mobility, court users, staff, contractors, deliveries and emergency responders.
- **Time horizon / status:** Current and recurring throughout the hold; asset-identification and documentation gap, not a verified failure.
- **Missing evidence:** Lift inventory and identifiers; examination certificates; risk assessments; emergency communication and rescue arrangements; fire-service classification; actual availability; identity of any second lift.
- **Sources:** [BetrSichV Annex 1](https://www.gesetze-im-internet.de/betrsichv_2015/anhang_1.html); [BetrSichV Annex 2](https://www.gesetze-im-internet.de/betrsichv_2015/anhang_2.html); [Official court page](https://ordentliche-gerichtsbarkeit.hessen.de/landgerichtsbezirk-frankfurt-am-main/amtsgericht-bad-homburg-v-d-hoehe); supplied lift records.

### R9. Water supply, hydrants, potable/fire-water separation and public-health exposure

- **New evidence:** DVGW identifies municipal and supplier responsibility for fire-water planning and hydrant capacity. Local water sources report demand pressure, purchased water and future dry-year supply concerns. TrinkwV testing and action provisions apply only if specified threshold conditions are met.
- **Existing property fact:** The property uses the public water network; two hydrants are recorded, a third is planned, potable and fire-water systems are not separated, and a separation station is reportedly being implemented at €77,000. Wet-riser, fire-water and public-health close-out records are incomplete.
- **Property linkage:** Public-water pressure or interruption, inadequate hydrant flow, defective separation or unverified hot-water conditions could affect firefighting, water hygiene and court continuity. No property-specific pressure, fire-flow shortfall or contamination event is established.
- **Likelihood / impact:** Medium for external supply pressure; Unknown for property-specific effect / High.
- **Affected:** Occupants, visitors, emergency services, fire protection, water systems and operations.
- **Time horizon / status:** Current through works completion and acceptance; dry-year external evidence extends toward 2030; emerging and conditional.
- **Missing evidence:** Hydrant flow and residual-pressure tests; fire-water demand; third-hydrant completion; separation design and commissioning; booster requirement; storage and pipe volumes; aerosol fixtures; Legionella sampling and authority records.
- **Sources:** [DVGW fire-water](https://www.dvgw.de/themen/wasser/netze-und-speicherung/loeschwasser?type=98); [Stadtwerke water](https://www.stadtwerke-bad-homburg.de/de/produkte/trinkwasser); [IWW water-management study](https://iww-wasserforschung.de/wassermanagement/); [TrinkwV §31](https://www.gesetze-im-internet.de/trinkwv_2023/__31.html); [TrinkwV §51](https://www.gesetze-im-internet.de/trinkwv_2023/__51.html); supplied water records.

### R10. Surface water, sewer surcharge, backflow and drainage integrity

- **New evidence:** Bad Homburg reports severe rainfall, flooded cellars and underground garages, raised manhole covers, ageing sewers and limitations in private drainage records. Municipal drainage approval processes include hydraulic, retention and overflow calculations.
- **Existing property fact:** The property has a below-street garage, steep driveway, DN1400 mixed-sewer connection, separator, unknown backwater level, no evidenced drainage tightness testing and incomplete backflow-protection records. A €41,500 testing and repair estimate is supplied.
- **Property linkage:** Intense rainfall or sewer surcharge → mixed sewer and uncertain private safeguards → backflow, leakage or uncontrolled discharge → garage water damage, access restriction, closure, repair cost or liability. No prior flood loss at the address is established.
- **Likelihood / impact:** Medium / High.
- **Affected:** Garage, vehicles, plant, electrical/mechanical equipment, users, access and operations.
- **Time horizon / status:** Current and recurring during intense rainfall; inferred and unresolved.
- **Missing evidence:** Parcel-level flood depth and flow paths; threshold elevations; as-built drainage drawings; tightness tests; legal backwater level; backflow protection tests; separator maintenance and approval; remediation evidence.
- **Sources:** [City Starkregen](https://bad-homburg.de/de/stadt/umwelt-und-klima/wasser/hochwasser-und-starkregen); [City Kanalnetz](https://bad-homburg.de/de/stadt/planen-und-bauen/stadtentwaesserung/kanalnetz); [City drainage approval](https://bad-homburg.de/de/stadt/rathaus/online-services/entwaesserungsgenehmigung-antrag-auf-vorpruefung~lOV8EzNr5Kn); [Hochtaunuskreis severe weather](https://www.hochtaunuskreis.de/Aktuelles/Starkregen-und-Unwetter-im-Hochtaunuskreis-Zusammenhalt-und-Einsatzbereitschaft-in-herausfordernden-Zeiten.php?ModID=255&FID=3419.20696.1)

### R11. Groundwater, hazardous liquids and protected-water liability

- **New evidence:** The site is within the Bad Homburg healing-spring protection area; AwSV imposes containment and inspection requirements for relevant hazardous-liquid installations. Public monitoring evidence does not establish parcel-specific groundwater levels.
- **Existing property fact:** Records report groundwater expected at excavation level, a watertight basement, historic heating-oil storage and hazardous-liquid installations without confirmed bunding or separator arrangements. No sampling was performed and no suspected contamination is reported in the supplied file.
- **Property linkage:** Groundwater or perched-water ingress could affect the below-grade structure. A hazardous-liquid release could migrate through soil or groundwater, producing investigation, remediation, regulatory, third-party and insurance exposure.
- **Likelihood / impact:** Unknown / High.
- **Affected:** Basement, garage, soil, groundwater, healing-spring resources, occupants, regulators, third parties and remediation economics.
- **Time horizon / status:** Current controls with long-tail liability; regulatory exposure in principle, no contamination or release established.
- **Missing evidence:** Exact protection-zone overlay; groundwater series and datum; tank location, type, capacity and substance class; containment and inspection records; drainage condition; soil and groundwater sampling; authority and claims records.
- **Sources:** [Hessian Geoportal healing-spring area](https://www.geoportal.hessen.de/spatial-objects/272/collections/inspire_bewirtschaftungsgebiete%3ATWS_HQS_ALK/items/TWS_HQS_ALK.4978?f=html); [AwSV](https://www.gesetze-im-internet.de/awsv/BJNR090500017.html); [AwSV Annex 6](https://www.gesetze-im-internet.de/awsv/anlage_6.html); [City groundwater](https://bad-homburg.de/de/stadt/umwelt-und-klima/wasser/grundwasser); supplied ground and water records.

### R12. Incomplete works, close-out, specialist capacity and equipment obsolescence

- **New evidence:** External labour data show long vacancy durations in relevant technical trades; Siemens and cited approval records show that some legacy product lines or approvals may be obsolete, although installed-model identity is not established.
- **Existing property fact:** Open fire-safety, garage, revolving-door, MSR, lighting and authority items remain incompletely documented. Records cite provider capacity constraints, a Chemicon recommendation, a pending FMC offer, unknown manufacturer delivery and missing TÜV or authority closure. The fire-alarm centre and other equipment may approach end of life.
- **Property linkage:** Specialist labour, approved-system compatibility and manufacturer delivery dependencies → delayed testing, re-tendering, replacement or acceptance → prolonged defects, garage or system downtime, duplicated fees and cost escalation.
- **Likelihood / impact:** Medium / Medium to High; exact asset-specific likelihood Unknown.
- **Affected:** Life-safety systems, garage, controls, lighting, court continuity, contractors and CapEx.
- **Time horizon / status:** Current through completion, with lifecycle exposure over 1–5 years; supplied delay and lifecycle exposure, external causation and final procurement status unresolved.
- **Missing evidence:** Executed appointments and contracts; model and serial-number schedule; parts availability; final offers and delivery commitments; critical path; TÜV and authority records; substitution and delay terms; completion and acceptance.
- **Sources:** [Federal Employment Agency handcraft data](https://statistik.arbeitsagentur.de/DE/Statischer-Content/Statistiken/Themen-im-Fokus/Berufe/Online-KV/Arbeits-und-Ausbildungsmarkt-im-Handwerk.html); [Siemens Cerberus](https://www.siemens.com/de-de/products/cerberus/); [DIBt approval Z-6.5-265](https://www.dibt.de/de/service/zulassungsdownload/detail/z-6.5-265); [ESSER IQ8Control](https://www.esser-systems.com/en/products/details/system-iq8control/iq8control-m/808004-facp-iq8control-m/); supplied works records.

### R13. Incomplete public-law approvals, lawful use and fire-safety closure

- **New evidence:** The City’s B-Plan 117 is informational unless matched to the official plan; current HBO, HBKG, GVSV, TPrüfV and GaV establish continuing approval, inspection and operation frameworks.
- **Existing property fact:** Records contain conflicting permits and dates for the original building, extension, dormers and attic works; missing acceptance documents; historical attic restrictions; incomplete fire-service, escape, rescue and fire-protection records; and unclosed deadlines.
- **Property linkage:** If existing uses or works cannot be matched to approvals and accepted as-built conditions, future alteration, financing, transfer or occupation could face additional documentation, consent, works or use restrictions. Current court operation is contrary evidence against a total prohibition.
- **Likelihood / impact:** Medium / High.
- **Affected:** Courtrooms, offices, attic, extension, garage, occupiers, emergency responders, financing and exit documentation.
- **Time horizon / status:** Current and 0–5 years, especially on works, refinancing or disposal; inferred approval-continuity risk, no enforcement order established.
- **Missing evidence:** Complete Bauakte; executed plan and parcel overlay; permits and drawings; deviations and conditions; acceptance and occupancy records; current use classification; fire-safety concept; authority correspondence; closure evidence.
- **Sources:** [City B-Plan 117](https://www.bad-homburg.de/de/stadt/planen-und-bauen/bebauungsplaene/bebauungsplan-nr-117~dz7AprMV5nk); [Hessian Building Code](https://wirtschaft.hessen.de/wohnen-und-bauen/baurecht-und-bautechnik/hessische-bauordnung-hbo); [§15 HBKG](https://www.rv.hessenrecht.hessen.de/bshe/?docId=jlr-NNLHE000050AENN00000000036&query=JURISLINK%3A%22Brand%2FKatSchG+HE+%C2%A7+15%22); [TPrüfV](https://www.rv.hessenrecht.hessen.de/bshe/?query=DOKNR%3Ajlr-NNLHE00005472&source=PermaLink); [GaV 2024](https://verkuendung.hessen.de/sites/verkuendung.hessen.de/files/veroeffentlichungsplattform_plugin/published/274/GVBl_2024_Nr_18_Regelungstext.pdf)

### R14. Title, cadastral identity, access rights and parking conformity

- **New evidence:** ALKIS, Grundbuch and Baulasten records are the relevant sources for parcel identity, ownership and successor-binding burdens; the municipal parking ordinance provides conditional calculation and payment-in-lieu mechanisms.
- **Existing property fact:** Records conflict on parcels, areas, cadastral identifiers, consolidation, postal code, 88 versus 89 spaces, access over municipal or other land and drainage, utility and fire-service rights.
- **Property linkage:** Unreconciled parcels and rights prevent reliable allocation of buildings, garage, access, parking, utilities and security scope. A missing or ineffective burden, easement or buyout could affect lawful use, emergency access, works authority, financing and transfer.
- **Likelihood / impact:** Unknown to Medium / High.
- **Affected:** All buildings, garage, court users, emergency services, owners, lenders and transaction economics.
- **Time horizon / status:** Current and at works, refinancing or disposal; unresolved governance and conformity risk, no title or access defect established.
- **Missing evidence:** Current Grundbuch; ALKIS records; consolidation and boundary records; Baulastenverzeichnis; private easements; access and fire-service plans; approved parking calculation; buyout decision; maintenance obligations.
- **Sources:** [Official court address](https://ordentliche-gerichtsbarkeit.hessen.de/landgerichtsbezirk-frankfurt-am-main/amtsgericht-bad-homburg-v-d-hoehe/wegbeschreibung); [HVBG cadastral outputs](https://hvbg.hessen.de/liegenschaftskataster/liegenschaftskataster-ausgaben); [§12 GBO](https://www.gesetze-im-internet.de/gbo/__12.html); [Bad Homburg Baulasten](https://www.bad-homburg.de/de/stadt/planen-und-bauen/bauaufsicht-und-bauordnung/baulasten); [Stellplatzsatzung](https://app.bad-homburg.de/stadt/rathaus/stadtrecht-aktuell/stadtrecht/dokumente/2022/Stellplatzsatzung_mit_Anlagen.pdf)

### R15. Owner, fund, KVG, agent and landlord authority chain

- **New evidence:** Public records identify active PATRIZIA entities and portfolio-level structures but do not connect them conclusively to the subject parcel. German corporate and fund-law representation depends on the applicable entity, general partner, KVG, depositary and authority documents.
- **Existing property fact:** Records name PATRIZIA Hessen Eins GmbH & Co. KG, PATRIZIA Real Assets KVG, PATRIZIA Immobilien KVG, PATRIZIA Deutschland, CA Immo, Res Publica portfolios and property management references, without a reconciled current titleholder or mandate.
- **Property linkage:** Works contracts, lease amendments, financing documents or disposals may be signed by an entity whose ownership or authority is not demonstrated, affecting enforceability, funding, reporting and transfer timing.
- **Likelihood / impact:** Unknown / High.
- **Affected:** Works, leases, financing security, fund compliance, repairs and exitability.
- **Time horizon / status:** Current and at all material works, financing or disposal events; established documentation gap, no invalid contract or authority failure established.
- **Missing evidence:** Land-register and commercial-register extracts; entity and general-partner details; fund classification; property schedule; KVG and depositary; powers of attorney; approval matrix; works and disposal authority.
- **Sources:** [LEI record](https://lei.bloomberg.com/leis/view/529900HM378OMB9B7K45); [§35 GmbHG](https://www.gesetze-im-internet.de/gmbhg/__35.html); [§170 HGB](https://www.gesetze-im-internet.de/hgb/__170.html); [§17 KAGB](https://www.gesetze-im-internet.de/kagb/__17.html); [LBBW Res Publica Hessen I](https://www.lbbw.de/artikel/pressemitteilung/patrizia-res-publica-hessen-i-portfolio_ahonfbexfr_d.html)

### R16. Lease tail, renewal and landlord-repair allocation

- **New evidence:** Hessian material identifies the address as LEO II object 3 with possession transfer on 1 January 2007 and a reported 25-year term. Portfolio-level materials refer to longer terms, State termination rights and negotiations to extend or waive options.
- **Existing property fact:** The court is the current operating user; no executed lease, amendment, break notice, renewal agreement, rent, recoveries or repair schedule is supplied. LEO summaries distinguish “Dach und Fach,” while the supplied assessment assumes tenant responsibility subject to exceptions without lease analysis.
- **Property linkage:** A break, non-renewal or changed repair allocation could produce vacancy, reletting, court-specific fit-out, landlord repair and unbudgeted capital exposure. Current use does not establish renewal.
- **Likelihood / impact:** Unknown / High for lease tail; Unknown / Medium for repair leakage.
- **Affected:** Income, vacancy, building works, specialist fit-out, court continuity, Opex and CapEx.
- **Time horizon / status:** Current information gap; nominal term timing around 2032 if the reported 2007/25-year reading applies, with broader mid-2030s alternatives; inferred, no termination event established.
- **Missing evidence:** Executed lease and amendments; commencement and expiry; break and renewal rights; rent and indexation; tenant works; relocation and reletting obligations; repair allocation; recoveries and current arrears.
- **Sources:** [Hessian Landtag Drucksache 16/6376](https://starweb.hessen.de/cache/DRS/16/6/06376.pdf); [Drucksache 16/7905](https://starweb.hessen.de/cache/DRS/16/5/07905.pdf); [Drucksache 20/10943](https://starweb.hessen.de/cache/DRS/20/3/10943.pdf); [Hessian LEO negotiations](https://vg-frankfurt.justiz.hessen.de/presse/pressearchiv/moderne-heimatnahe-und-nachhaltige-arbeitsplaetze-fuer-beschaeftigte); [OLG Frankfurt repair allocation](https://ordentliche-gerichtsbarkeit.hessen.de/presse/kein-vorschussanspruch-des-landes-hessen-gegen-vermieter-wegen-grossflaechiger-innenputzschaeden)

### R17. Energy baseline, automation and system-performance uncertainty

- **New evidence:** GEG certificate provisions require relevant non-residential consumption data; the earlier automation framework used a 290 kW threshold and the later enacted framework uses a 70 kW threshold from 29 July 2026 with a 31 December 2029 deadline, subject to applicability and exceptions.
- **Existing property fact:** The 2021 certificate contains conflicting end-energy values of 97 and 21 kWh/m²a against 143 primary energy; current bills, fuel split, remote reading, submetering, BMS/MSR configuration and emissions calculations are missing. Historic heating capacity is stated as 360 kW and controls are near end of life.
- **Property linkage:** Unverified energy performance and control capacity may affect energy cost, carbon reporting, monitoring compliance, plant operation and future technical CapEx. Current non-compliance is not established.
- **Likelihood / impact:** High for data gap; Medium for automation exposure / Medium.
- **Affected:** All buildings, heating, ventilation, lighting, controls, occupants and operating economics.
- **Time horizon / status:** Current; automation milestones include 31 December 2024 and conditionally 31 December 2029; established data gap and emerging regulatory exposure.
- **Missing evidence:** Complete certificate; three years of bills; meter map; current capacities; BMS architecture; data interfaces; lighting and indoor-climate controls; legal version and exceptions.
- **Sources:** [GEG §79](https://www.gesetze-im-internet.de/geg/__79.html); [GEG §82](https://www.gesetze-im-internet.de/geg/__82.html); [GEG §85](https://www.gesetze-im-internet.de/geg/__85.html); [GModG §56](https://www.gesetze-im-internet.de/geg/__56.html); [BGBl. 2026 I Nr. 226](https://www.recht.bund.de/bgbl/1/2026/226/VO.html); supplied energy records.

### R18. Heating-source contradiction and transition exposure

- **New evidence:** The future framework imposes renewable or low-carbon fuel shares on qualifying new fossil-fuel heating installations from 2029 onward; BEHG sets stated fuel-emission certificate prices for 2025–2026. WPG heat-planning deadlines and network rules are conditional.
- **Existing property fact:** Records conflict between district heating and two oil-fired boilers totalling approximately 360 kW, with uncertain gas infrastructure, controls, metering and district-heating interface.
- **Property linkage:** If fossil plant remains, replacement, carbon-cost and fuel-compliance exposure may arise; if district heating is actual, exposure depends on network tariff, emissions factor, contract and interface. Current breach or replacement duty is not established.
- **Likelihood / impact:** Unknown / Medium to High depending on configuration.
- **Affected:** Occupants, court operations, plant room, heating and hot water, Opex, CapEx and commissioning downtime.
- **Time horizon / status:** 2028–2040 and throughout the hold; conditional emerging regime exposure.
- **Missing evidence:** Current heat source, plant and fuel meters; district-heating contract and supplier; gas isolation; plant age and redundancy; emissions factor; heat-plan status; replacement timing.
- **Sources:** [GModG §43](https://www.gesetze-im-internet.de/geg/__43.html); [BEHG §10](https://www.gesetze-im-internet.de/behg/__10.html); [WPG §4](https://www.gesetze-im-internet.de/wpg/__4.html); [WPG §29](https://www.gesetze-im-internet.de/wpg/__29.html); [BMWSB existing-building guidance](https://www.bmwsb.bund.de/SharedDocs/faqs/DE/geg/04-was-gilt-in-zukunft-im-bestand.html?nn=42730); supplied heating records.

### R19. Lighting, charging and cooling-equipment lifecycle exposure

- **New evidence:** Certain T5/T8 fluorescent-lamp exemptions expired in 2023; GEIG includes conditional charging-point and conduit duties for qualifying non-residential parking; Regulation (EU) 2024/573 restricts placing specified high-GWP split systems on the market by equipment category and date.
- **Existing property fact:** Records identify predominantly T5/T8 lighting, a €42,000 LED estimate, underground parking with more than 20 spaces, unknown charging provision and two R410A split systems with unknown capacity and charge.
- **Property linkage:** Product availability, charging obligations or failed cooling equipment may require broader luminaire, electrical, controls or replacement works and temporary disruption. Applicability and cost are unknown.
- **Likelihood / impact:** Medium for lighting lifecycle; Unknown for charging and R410A / Medium.
- **Affected:** Courtrooms, offices, printing area, garage, electrical systems, users and CapEx.
- **Time horizon / status:** Current through 2035; emerging and conditional, no current breach established.
- **Missing evidence:** Lamp and ballast inventory; emergency-lighting compatibility; parking count by building; charging records; renovation scope and dates; electrical capacity; R410A model, charge, leak and service records.
- **Sources:** [Directive (EU) 2022/284](https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A32022L0284); [GEIG §9](https://www.gesetze-im-internet.de/geig/__9.html); [GEIG §10](https://www.gesetze-im-internet.de/geig/__10.html); [Regulation (EU) 2024/573](https://eur-lex.europa.eu/eli/reg/2024/573/2024-02-20/eng); [UBA building air-conditioning](https://www.umweltbundesamt.de/themen/klima-energie/fluorierte-treibhausgase-fckw/natuerliche-kaeltemittel-in-stationaeren-anlagen/anwendungen/gebaeudeklimatisierung)

### R20. Incomplete works budget, construction-price, procurement and funding exposure

- **New evidence:** Destatis reports a 5.2% year-on-year increase in German office construction prices from May 2025 to May 2026. Construction-sector insolvency and specialist-labour data support possible delivery pressure but do not identify failure of a property contractor.
- **Existing property fact:** The stated ten-year programme is €1,888,100 net, unindexed and undiscounted, with approximately 70% in years 2–5. Other totals and garage cost bases conflict; VAT, soft costs, contingencies, downtime, defects and whole-system implications are incomplete. No debt, reserve, valuation or covenant data is supplied.
- **Property linkage:** Price pressure and unresolved scope/procurement → additional works, VAT, professional fees, delay or re-tendering → liquidity, garage or court disruption and possible funding requirement. Effect on debt service or covenants is unknown.
- **Likelihood / impact:** Medium for market cost pressure; asset overrun likelihood Unknown / Medium; financing impact Unknown.
- **Affected:** Building renewal, garage, fire safety, owner or fund liquidity, court continuity and capital economics.
- **Time horizon / status:** Immediate through years 1–10, greatest stated spend in years 2–5; emerging and inferred, no quantified funding gap.
- **Missing evidence:** Reconciled cost plan; tender and contract prices; VAT recovery; soft costs; contingencies; final scope; contractor and delivery terms; cash, debt, reserves, valuation and covenant metrics.
- **Sources:** [Destatis construction-price index](https://www.destatis.de/EN/Themes/Economy/Short-Term-Indicators/Prices/bpr110.html); [§12 UStG](https://www.gesetze-im-internet.de/ustg_1980/__12.html); [§15 UStG](https://www.gesetze-im-internet.de/ustg_1980/__15.html); [Bundesbank lending survey](https://www.bundesbank.de/en/press/press-releases/january-results-of-the-bank-lending-survey-bls-in-germany-950082); supplied cost and procurement records.

### R21. Reletting, court-space change and exit-liquidity uncertainty

- **New evidence:** Bad Homburg office-market reports show limited take-up, significant available space and differing rent and vacancy measures; market data indicate stronger demand for modern, high-quality space. E-files and jurisdiction changes may alter court-space requirements.
- **Existing property fact:** The asset is an older, specialist courthouse with cells, archives, hearing rooms, secure circulation and conflicting area records. No current lease, occupancy, room use, property-specific rent, valuation or alternative-use approval is supplied.
- **Property linkage:** If judicial occupation reduces or ends, specialist configuration and unresolved lawful-use, area, access and condition records may narrow the alternative-tenant and buyer pool, increasing vacancy, fit-out, incentives, valuation dispersion and exit timing.
- **Likelihood / impact:** Medium for alternative-use demand; Unknown for court-space change / High.
- **Affected:** Court operations, alternative tenants, vacancy, rent, fit-out, buyer and lender underwriting and exit economics.
- **Time horizon / status:** Current market context; material on any vacancy during the ten-year hold; inferred, no current vacancy or relocation established.
- **Missing evidence:** Executed lease and renewal status; occupancy and room utilisation; verified areas; conversion permissions; fit-out and vacancy costs; court staffing and case data; property-specific comparables and valuation.
- **Sources:** [NAI Bad Homburg office market](https://nai-apollo.de/marktberichte/bueromaerkte-im-frankfurter-umland-2024/); [IHK Gewerbemarktbericht 2025](https://www.frankfurt-main.ihk.de/branchenthemen/bau-und-immobilienwirtschaft/gremienarbeit-politikberatung/frankfurter-immobilienboerse/publikationen-der-frankfurter-immobilienboerse/gewerbemarktbericht-2025-5326540); [City Immobilienmarktbericht](https://www.bad-homburg.de/de/stadt/aktuelles/der-immobilienmarktbericht-2026-n28g44v5jg); [ImmoWertV](https://www.gesetze-im-internet.de/immowertv_2022/BJNR280500021.html); [Hessen Justice e-file announcement](https://justizministerium.hessen.de/presse/e-akte-erfolgreich-in-der-justiz-eingefuehrt)

### R22. Insurance and residual uninsured-loss exposure

- **New evidence:** German insurance guidance distinguishes standard storm/hail cover from heavy-rain, flooding and backflow cover; groundwater and backflow conditions may be excluded or warranty-dependent. Public sources do not disclose the property’s ZÜRS class or policy terms.
- **Existing property fact:** The property has a below-grade garage, drainage and backflow uncertainties, hazardous-liquid exposure and a €41,500 drainage estimate. No policy, claims history, limits, exclusions or business-interruption schedule is supplied.
- **Property linkage:** Physical exposure plus unresolved safeguards → insurer-specific exclusions, deductibles, limits or warranties may affect repair, clean-up, liability and interruption costs. No coverage gap or insurer refusal is established.
- **Likelihood / impact:** Unknown / High.
- **Affected:** Building, garage, clean-up, liability, business interruption, owner liquidity and insurability.
- **Time horizon / status:** Current and at renewal; materially unresolved.
- **Missing evidence:** Policy wording and schedule; insured perils; limits and sublimits; deductibles; warranties; natural-hazard and environmental extensions; business interruption; claims history; renewal terms.
- **Sources:** [GDV elemental insurance](https://www.gdv.de/gdv/themen/klima/fragen-und-antworten-zur-debatte-um-eine-elementarschaden-pflichtversicherung--202354); [Verbraucherzentrale](https://www.verbraucherzentrale.de/wissen/geld-versicherungen/weitere-versicherungen/versicherungsschutz-gegen-elementarschaeden-11440); [Hessen insurance guidance](https://landwirtschaft.hessen.de/wasser/hochwasserschutz/versicherung); [GDV flood check](https://www.dieversicherer.de/versicherer/wohnen/hochwassercheck)

## Effects on the building, people and operations

- **Life safety and evacuation:** Fire-door defects, uncertain compartmentation, RWA contradictions, alarm audibility, transmission-route uncertainty and common electrical dependencies can each affect warning, smoke movement, escape and fire-service response. They remain distinct causes with a shared potential effect; no fire event or unsafe total-building condition is established.
- **Garage safety and availability:** Concrete corrosion, gate failure, uncertain ventilation/CO capacity, drainage/backflow exposure, surface-water entry, groundwater uncertainty and external parking restrictions may separately reduce garage safety, access or availability. No current structural instability, CO event or property flood loss is established.
- **Accessibility and public access:** Lift documentation gaps, potentially concentrated lift dependency, steep or constrained approaches, parking uncertainty and dated transport works may affect persons with reduced mobility, court visitors, staff, deliveries and emergency vehicles. The official court page provides contrary evidence of a ramp, passenger lift, bus access and stated parking.
- **Building services continuity:** Electrical protection, emergency power, MSR, heating, ventilation, cooling, lighting, water and fire-water systems may experience outage, compatibility or commissioning effects. The evidence establishes incomplete assurance and lifecycle exposure, not current system-wide failure.
- **Court operations:** Works, inspection, acceptance, garage closure, access disruption, heating or water interruption and changes to archive, hearing-room or office requirements may affect court continuity, but no closure, relocation or quantified downtime is evidenced.
- **People and environmental exposure:** Potential CO accumulation, fire, water contamination, hazardous-liquid release, heat stress, dampness and flood-related hazards affect occupants, visitors, contractors, emergency responders, soil and groundwater. Actual incidents are not established at the property.
- **Economics and income:** Unresolved lease, repair allocation, works scope, cost escalation, insurance and funding evidence may affect owner leakage, cash requirements and income continuity. No arrears, rent abatement, debt distress or quantified income loss is established.
- **Reletting and exit:** If court occupation changes, specialist configuration, uncertain lawful use, access, areas, parking and condition may affect alternative demand, fit-out, vacancy, valuation and buyer depth. This remains conditional and is not evidence of current vacancy.

## Contradictions and material unknowns

### Competing readings that remain unresolved

- **RWA:** One 2025 report says systems are functional and safe for continued operation; another reports material safety concerns and reinspection. The underlying reports, scope and operating conditions are missing.
- **Fire alarm:** One record reports malfunction and insufficient audibility in rooms 103 and 105; another reports no defect.
- **Heating:** Records alternate between district heating and two 360 kW oil-fired boilers, with conflicting gas-removal, residual-piping and isolation descriptions.
- **Energy certificate:** Primary energy is stated as 143 kWh/m²a, while end-energy values of 97 and 21 kWh/m²a are both recorded.
- **Garage airflow:** 7,860 m³/h is described as required airflow, but it is unknown whether it is design, actual, historic or measured capacity.
- **Garage costs:** Cost bases of €765,000 and €702,668.06 conflict; the relationship to the €1,888,100 ten-year programme is unresolved.
- **Works status:** Some works are described as complete or substantially complete, but acceptance, commissioning, warranties and authority close-out are absent.
- **Utilities:** Records state that electricity, gas, drinking water and fire-water supplies are secured, while capacity, connection, isolation and redundancy remain incomplete.
- **Lift count and function:** Supplied records identify two lifts in one reading and one combined lift in another; the official court page confirms one passenger lift.
- **Parking:** Records report 89 required and 88 existing spaces, while other schedules provide different internal and external counts; buyout, burdens and access rights are unresolved.
- **External parking:** The court directions page describes free Seedammweg parking; current municipal evidence records restricted use and a €50 charge for non-bath users. Whether the facilities are the same is not explicit.
- **Address and cadastral identity:** Postal codes 61325 and 61352, parcel references 122/11 and 123/1 versus 122/1, 122/7 and 125/1, areas of 1,145 m² and 5,216 m², and cadastral identifiers 06798 and 04798 conflict.
- **Parcel consolidation:** A 1986 union is reported, while other records refer to an application or not-yet-entered plan data.
- **Planning and approvals:** Records refer to historic plans and §107 HBO conditions as well as B-Plan 117 and current HBO; their relationship to the courthouse parcel and current use is unresolved.
- **Fire-safety closure:** Records state both that a project was not technically fire-reviewed and that fire-alarm, fire-plan and inspection requirements applied. Deadlines through 2022 lack closure evidence.
- **Water protection:** Records refer to “no procedure,” “no prohibition affected” and “exception approval required.” External material distinguishes drinking-water Zone III from healing-spring quantitative Zones C and D.
- **Ownership and authority:** PATRIZIA, CA Immo, KVG, fund, Fraport and property-management references are not reconciled to current title, landlord, fund or signatory authority.
- **Lease term:** A 25-year term from 2007 implies approximately 2032, while broader LEO material refers to terms and expiries extending into the mid-2030s. No executed lease is supplied.
- **Market evidence:** Rents, vacancy and take-up figures differ by provider, geography, period and definition and cannot be combined into one property-specific trend.
- **Regulatory versions:** The earlier GEG §71a framework and later GModG §56 framework have different dates and thresholds; the applicable version depends on the relevant date and facts.
- **Technical identity:** Siemens, Minimax, Esser, Dorma/GEZE and other legacy references do not establish the exact installed model, approval status or replacement pathway.
- **Fire-alarm endpoint:** Records refer variously to a leased line, Siemens control centre and fire-brigade control centre; the current approved endpoint and redundant route are unknown.

### Material unknowns

1. Current registered owner, landlord, tenant, fund, KVG, depositary and authority chain.
2. Definitive parcels, boundaries, areas, consolidation, easements, Baulasten and municipal-land rights.
3. Complete lease, amendments, rent, recoveries, term, breaks, renewal and repair allocation.
4. Complete permits, approved drawings, acceptance, occupancy and historic-condition close-out.
5. Current fire-safety concept, door register, RWA, alarm, emergency-power, electrical, fire-water and lift certificates.
6. Structural capacity, corrosion extent, garage repair scope, gate programme and completion evidence.
7. Garage statutory area, traffic classification, airflow, CO readings, fan changeover and safety-power performance.
8. Drainage as-built condition, backwater level, tightness tests, backflow protection and separator approval.
9. Parcel-specific surface-water flow, threshold elevations, groundwater series and waterproofing condition.
10. Hazardous-liquid inventory, tank condition, containment, inspections, sampling and authority records.
11. Actual heating source, plant capacity, district-heating connection, gas isolation, controls, redundancy and fuel data.
12. Current energy bills, certificate calculation boundary, BMS, submetering, lighting, ventilation, cooling and fabric performance.
13. Parking count, allocation, accessible spaces, emergency routes, buyout and external parking dependency.
14. Existing charging infrastructure, electrical capacity and qualifying renovation dates.
15. R410A equipment capacity, charge, service and leak history.
16. Reconciled works scope, costs, VAT treatment, soft costs, contingencies, contractor appointments, delivery dates and downtime.
17. Debt, lender, maturity, rates, hedge, security, reserves, valuation and covenant metrics.
18. Insurance wording, limits, deductibles, exclusions, claims and business-interruption cover.
19. Court staffing, case volumes, room utilisation, archive needs, e-file implementation and site-specific occupation plans.
20. Exact start date of the stated ten-year hold.
21. No named domain response was empty in the supplied material. Any fact not supplied within a domain remains unknown rather than being treated as resolved.

## No material pathway established

The following researched subjects remained context-only or conditional because the required property linkage was not established:

- **Permanent courthouse closure, relocation or redevelopment:** Official sources confirm continuing court operations, but no property-specific closure, relocation or redevelopment decision was evidenced.
- **Fulda register-court reform:** The planned 2030 register court establishes an institutional change but not a material reduction in occupation, staffing or income at Bad Homburg.
- **District-heating projects and municipal heat networks:** Local heat, wastewater and network projects do not establish that this property is connected to, included in or dependent on a qualifying network. [Stadtwerke heat/cooling systems](https://www.stadtwerke-bad-homburg.de/de/produkte/waerme-kaelteanlage); [City wastewater-heat report](https://www.bad-homburg.de/de/stadt/aktuelles/baufortschritt-an-der-klaranlage-ober-eschbach-xlyovnv5v9); [WPG §23](https://www.gesetze-im-internet.de/wpg/__23.html)
- **EPBD worst-performing-building thresholds and Hessian PV duties:** The property’s energy ranking, ownership classification, roof status, parking configuration, implementation rules and exemptions are unknown. [Directive (EU) 2024/1275](https://eur-lex.europa.eu/eli/dir/2024/1275/oj); [HEG §9a](https://www.rv.hessenrecht.hessen.de/bshe/?query=DOKNR%3Ajlr-NNLHE000053C0NN00000000022&source=PermaLink); [HEG §12](https://www.rv.hessenrecht.hessen.de/bshe/?query=DOKNR%3Ajlr-NNLHE000053C0NN00000000032&source=PermaLink)
- **Regional ground movement, settlement, earthquake and radon:** Regional geological and radon layers do not establish parcel-specific movement, structural damage or indoor exposure.
- **Unexploded ordnance:** No property-specific assessment or clearance result for Auf der Steinkaut 10–12 was located.
- **Monument, tree and landscape protection:** No protected monument, qualifying tree or relevant property-specific approval was established. [Tree-protection rules](https://www.bad-homburg.de/de/stadt/umwelt-und-klima/natur-und-landschaft/baumschutzsatzung)
- **Nearby contamination and fires:** The ALTIS reference at Auf der Steinkaut 25 and a container or gymnasium-related fire do not establish contamination, migration, fire or spread at the courthouse.
- **General transport, cycle-route and planning proposals:** Beyond the dated developments identified above, no confirmed acquisition, easement, closure or construction effect at the courthouse was established.
- **Public-procurement breach:** Public court use and a public tenant do not establish that the private owner, KVG or SPV was a contracting authority; no breach or challenge was identified.
- **Specific tax liability:** Grundsteuer and Grunderwerbsteuer regimes are relevant in principle, but no assessment, error, taxable transaction, exemption or amount is established. [Hessian Grundsteuer B](https://finanzamt.hessen.de/grundsteuer/grundsteuer-b-in-hessen); [§1 GrEStG](https://www.gesetze-im-internet.de/grestg_1983/__1.html)
- **Portfolio refinancing and debt:** LBBW portfolio financing does not identify this address and does not establish property-level debt, security, maturity, covenant or refinancing exposure.
- **Foreign exchange, sanctions, trade routes and critical materials:** No foreign-currency debt, sanctioned counterparty, country of origin, restricted commodity, supplier route or critical-material dependency is linked to the property. [Regulation (EU) 2025/395](https://eur-lex.europa.eu/eli/reg/2025/395/oj); [Critical Raw Materials Act](https://eur-lex.europa.eu/eli/reg/2024/1252/oj/eng)
- **Cyber incident or property-specific telecommunications outage:** The records identify information gaps and technical dependencies, but no incident, outage or verified cyber weakness.
- **Water-supply restriction at the property:** Citywide water stress and a separate water-main failure do not establish inadequate pressure, interruption or fire-flow deficiency at this building.
- **General market comparables and conversion examples:** Other courthouse conversions, regional rents, vacancy, yields and public-sector demand evidence do not establish conversion feasibility, current value, rent, buyer depth or exit timing for this asset.